# CryptoCurrency NLP Sentiment Analyser — Exploratory Analysis

This notebook demonstrates the full NLP pipeline:
1. Data ingestion from CoinGecko
2. Text preprocessing
3. VADER + FinBERT sentiment analysis
4. Model evaluation (F1, confusion matrix)
5. Visualisation of sentiment vs. price correlation

**Tech stack:** Python · VADER · HuggingFace FinBERT · scikit-learn · pandas · matplotlib

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.ingestion import CoinGeckoClient, NewsIngestion, TRACKED_COINS
from models.sentiment_model import CryptoSentimentAnalyser
from models.evaluation import evaluate_model, build_labeled_dataset

print('✓ Imports successful')

## 1. Load NLP Models
Loads VADER with crypto domain extension and FinBERT (optional).

In [ ]:
# use_finbert=False for fast demo; set True for full accuracy
analyser = CryptoSentimentAnalyser(use_finbert=False)
print('Models loaded.')

## 2. Fetch Live Prices from CoinGecko

In [ ]:
gecko = CoinGeckoClient()
prices = gecko.get_prices()
prices[['symbol','name','price_usd','change_24h_pct','change_7d_pct','market_cap_usd']].head(8)

## 3. Run Sentiment Analysis on News Headlines

In [ ]:
news_client = NewsIngestion()
coin = TRACKED_COINS[0]  # Bitcoin

news = news_client.get_news(coin['name'], coin['symbol'])
texts = [f"{n['title']} {n['body']}" for n in news]

results = analyser.analyse_batch(texts)

# Display results as DataFrame
df_results = pd.DataFrame([
    {
        'headline': r.text[:80],
        'vader':    r.vader_compound,
        'ensemble': r.ensemble_score,
        'label':    r.label,
        'signal':   r.signal,
        'keywords': ', '.join(r.keywords[:3]),
    }
    for r in results
])
df_results

## 4. Aggregate Sentiment Per Coin

In [ ]:
all_summaries = []
for coin in TRACKED_COINS[:5]:
    news = news_client.get_news(coin['name'], coin['symbol'])
    texts = [f"{n['title']} {n['body']}" for n in news]
    results = analyser.analyse_batch(texts)
    summary = analyser.aggregate(results)
    all_summaries.append({'coin': coin['symbol'], **summary})

summary_df = pd.DataFrame(all_summaries)
summary_df[['coin','score','label','signal','confidence','positive_pct','negative_pct']]

## 5. Model Evaluation — Accuracy, F1, Confusion Matrix

In [ ]:
labeled_data = build_labeled_dataset()
report = evaluate_model(analyser, labeled_data)
report.print_summary()

## 6. Visualise Sentiment vs Price

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coins_plot = summary_df['coin'].tolist()
scores     = summary_df['score'].tolist()
colors     = ['#22c55e' if s > 0.05 else '#ef4444' if s < -0.05 else '#94a3b8' for s in scores]

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0f172a')
ax.set_facecolor('#1e293b')
ax.barh(coins_plot, scores, color=colors, edgecolor='none', height=0.5)
ax.axvline(0, color='white', linewidth=1)
ax.set_xlabel('Sentiment Score', color='white')
ax.set_title('Crypto Sentiment Dashboard', color='white', fontweight='bold')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
plt.tight_layout()
plt.show()